<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/main/module-09-multimodal-and-pretrained/lesson-9.1-multimodal/notebooks/GCP_Capstone_9.1_Multimodal.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 9.1 Gemini Multimodal — The Corpus's Own Images and Scans, Read and Checked
**Netsetos GenAI Engineering — GCP Capstone** · Module 9 · rebuilt on the live lane, 9 September 2026

Gemini reads images, PDFs and video by reference. Every input in this notebook is the lane's own: the invoice the corpus holds as text, rendered as a page image, so the extraction is checked against the truth; the one scan in the corpus (the POSH Act, 13 pages, no text layer), read directly and then through the lane; Figure 3 of the annual report, captioned twice - once by you, once by the ingest worker - and only one of those captions is the figure's door. Tokens are measured, not tabulated.


## Setup
The kit is cloned and `deploy/` is on the path, so `from shared import documind_tools` is the same import the API, the MCP server and the chat service make. The identity is the roster member `documind-ui-sa`, minted per call.


In [ ]:
!pip install -q google-genai==2.22.0 google-cloud-storage==3.13.1 Pillow==12.3.0 requests==2.34.2

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS: the project the lane runs in (make up, lesson 4.8)
REGION     = "us-central1"
TENANT     = "acme"
KIT        = "/content/agentic-ai-weekend-gcp-learners"   # the kit: deploy/shared is the tool layer every lesson on the lane imports
BRANCH     = "main"        # the learner repo's branch: the notebooks and the kit (deploy/) ship there together

import os, subprocess, sys
import google.auth
from google.auth.transport.requests import AuthorizedSession
from google import genai
from google.genai import types

if not os.path.isdir(KIT):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "-b", BRANCH,
                    "https://github.com/netsetos/agentic-ai-weekend-gcp-learners", KIT], check=True)
sys.path.insert(0, f"{KIT}/deploy")                   # `from shared import ...` - the same layer every service imports

# The lane's URLs are deterministic: service name + project NUMBER (eventarc.tf builds them the same way).
creds, _ = google.auth.default()
NUMBER = AuthorizedSession(creds).get(
    f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}").json()["projectNumber"]
API_URL       = f"https://documind-api-{NUMBER}.{REGION}.run.app"
UPLOAD_BUCKET = f"{PROJECT_ID}-uploads"     # storage.tf: the bucket eventarc.tf watches - the corpus, media included
MEDIA_BUCKET  = f"{PROJECT_ID}-media"       # storage.tf: generated assets, 30-day lifecycle (a cache, not a record)
os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": "global",              # Gemini 3.x generation is served from the global endpoint
    "GOOGLE_GENAI_USE_VERTEXAI": "TRUE",
    "DOCUMIND_PROFILE": "gcp",
    "RAG_API_URL": API_URL,
    "RAG_TIMEOUT_S": "90",                          # 7.2's finding: a cold API takes longer than the default 20 s
    # A notebook has no metadata server to be anyone with: the kit mints its ID tokens AS this roster
    # member (7.1). On Cloud Run the service's own account is the identity and nothing is set.
    "DOCUMIND_IMPERSONATE_SA": f"documind-ui-sa@{PROJECT_ID}.iam.gserviceaccount.com",
})
MEMBER_SA   = os.environ["DOCUMIND_IMPERSONATE_SA"]
OUTSIDER_SA = f"documind-outsider-sa@{PROJECT_ID}.iam.gserviceaccount.com"   # IAM admits it, no roster does (4.8, 7.2)

from shared import documind_tools    # THE one retrieve(). Imported, never pasted - the contract gate fails a paste.
gen = genai.Client(enterprise=True, project=PROJECT_ID, location="global")   # every generate_content in this lesson

print("kit:", KIT, "| API:", API_URL, "| media:", f"gs://{MEDIA_BUCKET}")


## Cell 1: The corpus's media
Two buckets and one rule: the uploads bucket is the corpus (one prefix per tenant, the notification the worker listens to); the media bucket is a cache. The helpers list what is there, and the notebook refuses to continue without media - nothing below can be checked against an empty corpus.


In [ ]:
import json, requests, time
from google.cloud import storage

# THE ROUTES 9.4's Studio stands on, called the way the UI calls them: one ID token per request,
# minted AS the roster member, audience = the API (7.3's hour-long fuse never arms). The body names
# the tenant; the API checks the caller's email on that tenant's roster before it spends a paisa.
def api(path: str, body: dict | None = None, timeout: int = 120) -> tuple[int, dict | str]:
    """POST one API route as documind-ui-sa. Returns (status, json-or-text) - never raises on 4xx,
    because a refusal is data this module reads (the outsider cells)."""
    r = requests.post(f"{API_URL}{path}", json=body,
                      headers={"Authorization": f"Bearer {documind_tools._id_token(API_URL)}"}, timeout=timeout)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]

# The corpus's media objects, read as YOU (the Colab credential is a project owner; the lane's
# services read them as their own accounts). One client, both buckets.
gcs = storage.Client(project=PROJECT_ID)

def media_objects(prefix: str = f"{TENANT}/") -> list[str]:
    """Every image, video or recording under the tenant's prefix of the uploads bucket."""
    return sorted(b.name for b in gcs.list_blobs(UPLOAD_BUCKET, prefix=prefix)
                  if b.name.lower().endswith((".png", ".jpg", ".jpeg", ".mp4", ".mp3")))

def gcs_bytes(uri: str) -> bytes:
    bucket, _, name = uri.removeprefix("gs://").partition("/")
    return gcs.bucket(bucket).blob(name).download_as_bytes()

FIG3  = f"gs://{UPLOAD_BUCKET}/{TENANT}/annual_report_2026_fig3.png"
INV   = f"gs://{UPLOAD_BUCKET}/{TENANT}/inv_2026_0412.png"
PAGE  = f"gs://{UPLOAD_BUCKET}/{TENANT}/payment_of_bonus_act_1965_p30.png"
VIDEO = f"gs://{UPLOAD_BUCKET}/{TENANT}/townhall_2026_q1.mp4"
POSH  = f"gs://{UPLOAD_BUCKET}/{TENANT}/posh_act_2013.pdf"

have = media_objects()
HAS_VIDEO = f"{TENANT}/townhall_2026_q1.mp4" in have
print("media in the corpus:", have or "NONE - run `make media` and `make ingest-corpus` from deploy/ (README: media is a document)")
print("video:", "present" if HAS_VIDEO else "absent (make media MEDIA_ARGS=--video, or drop a recording in) - the video cells will say so")
assert have, "no media under the tenant's prefix: nothing in this lesson can cite a figure until the corpus holds one"


## Cell 2: The invoice as an image, checked against the text
The extraction is structured (a schema, not prose), and it is checked: the total the model reads off the page must equal the total the lane retrieves from the same invoice's text.


In [ ]:
from pydantic import BaseModel

# THE INVOICE, AS AN IMAGE. inv_2026_0412.png is the corpus's own invoice rendered as a page (make
# media, evals/build_media.py). The same document is in the corpus as TEXT, so the extraction can be
# CHECKED - against what retrieve() says the total is (golden row lk-10) - instead of admired.
class InvoiceFields(BaseModel):
    invoice_no: str
    date: str
    total_payable: str      # exactly as printed, e.g. "Rs 1,84,500"
    gst_amount: str

r = gen.models.generate_content(
    model="gemini-3.6-flash",
    contents=[types.Part.from_uri(file_uri=INV, mime_type="image/png"),   # a reference: the bytes stay in GCS
              "Read this invoice. Return the invoice number, the date, the total payable and the GST amount exactly as printed."],
    config=types.GenerateContentConfig(response_mime_type="application/json", response_schema=InvoiceFields))
fields = r.parsed
print(fields)

truth = documind_tools.retrieve("What is the total payable on invoice INV-2026-0412?", tenant_id=TENANT, brain="direct")
digits = lambda s: "".join(ch for ch in s if ch.isdigit())
assert digits(fields.total_payable) == "184500", fields.total_payable
assert "184500" in digits(truth.get("answer") or ""), truth.get("answer")
print("\nOCR total", fields.total_payable, "== the corpus's text: the image and the text agree, and the lane cited",
      [c["source_uri"].rsplit("/", 1)[-1] for c in truth["citations"]][:2])


## Cell 3: Tokens, measured
`count_tokens` on the real page image and on the real 13-page scan. The rule of thumb is 258 per tile and 258 per PDF page; the measured number is the one you budget with.


In [ ]:
# TOKENS, MEASURED - not tabulated. An image up to 384 px is 258 tokens; a larger one is tiled at 258
# per 768-px tile (2.1). A PDF page is an image to the model: 258 each. Measure both on REAL assets -
# the invoice page and the one scan in the corpus (the POSH Act, 13 pages, no text layer).
img_tokens = gen.models.count_tokens(model="gemini-3.6-flash",
    contents=[types.Part.from_uri(file_uri=INV, mime_type="image/png"), "Read this invoice."]).total_tokens
pdf_tokens = gen.models.count_tokens(model="gemini-3.6-flash",
    contents=[types.Part.from_uri(file_uri=POSH, mime_type="application/pdf"), "Summarise."]).total_tokens
POSH_PAGES = 13                                # evals/real_sources.json: text_layer false
FLASH_IN = 1.50 / 1_000_000                    # USD per input token, gemini-3.6-flash standard rate

print(f"invoice PNG 1240x1754   : {img_tokens:>6} tokens  (~{img_tokens // 258} tiles of 258)")
print(f"POSH Act scan, 13 pages : {pdf_tokens:>6} tokens  = {pdf_tokens / POSH_PAGES:.0f} per page (the rule of thumb says 258)")
print(f"to READ the scan once   : ${pdf_tokens * FLASH_IN:.5f}")
print("the lane read it ONCE, at ingest, with Document AI (4.1) - every question since has paid for a chunk, not for 13 pages")
assert POSH_PAGES * 150 < pdf_tokens < POSH_PAGES * 700, pdf_tokens


## Cell 4: The scan, two ways
Gemini on the PDF pays for 13 pages per question and cites nothing. The lane paid once, at ingest, and cites the page.


In [ ]:
# THE SCAN, TWO WAYS. Gemini reads the 13-page scan directly: every question pays for 13 pages again,
# and there is no citation, only prose. The lane read it once (Document AI at ingest, 12.5) and
# retrieves the chunk: the question pays for a chunk, and the answer names the page. Same Act.
Q = "Under this Act, what body must an employer constitute at every office or branch with ten or more workers?"
direct = gen.models.generate_content(model="gemini-3.6-flash",
    contents=[types.Part.from_uri(file_uri=POSH, mime_type="application/pdf"), Q + " Answer in one sentence."])
print("Gemini on the scan :", direct.text.strip()[:220])

lane = documind_tools.retrieve(Q, tenant_id=TENANT, brain="direct")
print("the lane           :", (lane.get("answer") or "")[:220])
print("cited              :", [f"{c['source_uri'].rsplit('/', 1)[-1]} p.{c.get('page')}" for c in lane.get("citations", [])][:3])
assert "internal committee" in (direct.text + (lane.get("answer") or "")).lower()
assert any("posh_act_2013" in c["source_uri"] for c in lane.get("citations", [])), \
    "the lane did not cite the scan - was posh_act_2013.pdf ingested through Document AI? (make ingest-corpus)"


## Cell 5: Where media lives
No `create_bucket`. The lane's Terraform made both buckets; this cell reads them.


In [ ]:
# WHERE MEDIA LIVES. Two buckets, one rule. The UPLOADS bucket is the corpus: one prefix per tenant,
# and the object.finalized notification eventarc.tf put on it wakes the ingest worker - for a PDF and
# for a PNG alike (media is a document). The MEDIA bucket holds generated assets and crops under a
# 30-day lifecycle: a cache, never a record. Nothing here creates a bucket. storage.tf made both, and
# a notebook that creates buckets in a project it does not own is how you end up with two of them.
for uri in (FIG3, INV, PAGE, VIDEO):
    name = uri.rsplit("/", 1)[-1]
    print(f"{'present' if f'{TENANT}/{name}' in have else 'absent ':7} {uri}")

b = gcs.get_bucket(MEDIA_BUCKET)
rule = next(iter(b.lifecycle_rules), {})
print("\nmedia bucket:", b.location, "| lifecycle:", rule.get("action", {}).get("type"), "after", rule.get("condition", {}).get("age"), "days")
generated = [x.name for x in gcs.list_blobs(MEDIA_BUCKET, prefix=f"{TENANT}/gen/")]
print(f"generated under {TENANT}/gen/ so far: {len(generated)} (9.2 and 9.4 add to this; the 30-day rule empties it)")


## Cell 6: Two captions, one asset
A caption and alt text in one structured call - and then the caption that counts, read back through `retrieve()`.


In [ ]:
# CAPTION + ALT TEXT IN ONE STRUCTURED CALL - and then the caption that COUNTS. The worker wrote its
# own caption for this figure at ingest ("describe this figure for retrieval", 12.5), embedded it, and
# THAT text is what retrieve() matches a question against. Yours is for a catalogue and a screen
# reader. The retriever's is the figure's only door: a figure whose caption misses the number cannot
# be found by the number.
class MediaDescription(BaseModel):
    caption: str        # one sentence, for a human scanning a list
    alt_text: str       # what a screen reader says; never starts with "image of" (it announces "image" first)
    doc_type: str       # invoice | receipt | contract | chart | table | other
    contains_pii: bool  # 12.6's guard cares; so does the DPDP story

d = gen.models.generate_content(model="gemini-3.6-flash",
    contents=[types.Part.from_uri(file_uri=FIG3, mime_type="image/png"),
              "Describe this figure for a catalogue and for a screen reader."],
    config=types.GenerateContentConfig(response_mime_type="application/json", response_schema=MediaDescription)).parsed
print("caption :", d.caption)
print("alt text:", d.alt_text)
print("type    :", d.doc_type, "| PII:", d.contains_pii)
assert not d.alt_text.lower().startswith("image of")

hits = documind_tools.retrieve("Figure 3: revenue by region, FY2025 versus FY2026", tenant_id=TENANT, top_k=8, brain="direct")
fig = next((c for c in hits.get("citations", [])
            if c.get("kind") == "figure" and c["source_uri"].endswith("annual_report_2026_fig3.png")), None)
assert fig, f"no figure citation for Figure 3 - kinds seen: {sorted({c.get('kind', 'text') for c in hits.get('citations', [])})}"
print("\nthe retriever's caption (the worker's, written at ingest):")
print("  ", fig["quote"][:320])
print("kind:", fig["kind"], "| media_url:", fig["media_url"])
assert "EMEA" in fig["quote"] or "91" in fig["quote"], "the worker's caption does not carry the table's numbers"


## Cell 7: Video by URI
Gated on the town hall being in the corpus. The cell says what to run if it is not.


In [ ]:
# VIDEO BY URI, AT LOW RESOLUTION. Never read a video into memory to send it: Part.from_uri passes a
# reference and the bytes stay in GCS. media_resolution is the cost dial most people never touch: LOW
# samples fewer tokens per frame and is right for "what was said and roughly when"; HIGH is for
# reading text off slides. 9.4 builds the whole segment pipeline; here, one question and its time.
if HAS_VIDEO:
    v = gen.models.generate_content(model="gemini-3.6-flash",
        contents=[types.Part.from_uri(file_uri=VIDEO, mime_type="video/mp4"),
                  "At what time (MM:SS) does a speaker say EMEA revenue fell, and by what percentage? One line."],
        config=types.GenerateContentConfig(media_resolution=types.MediaResolution.MEDIA_RESOLUTION_LOW))
    print(v.text.strip()[:200])
    vt = gen.models.count_tokens(model="gemini-3.6-flash",
                                 contents=[types.Part.from_uri(file_uri=VIDEO, mime_type="video/mp4")]).total_tokens
    print(f"video tokens at the default resolution: {vt:,}  (the old table said ~18,000 a minute; this is THIS video, measured)")
    assert "5.2" in v.text, v.text
else:
    print("no townhall_2026_q1.mp4 in the corpus. From deploy/: make media MEDIA_ARGS=--video (Cloud Shell has ffmpeg and")
    print("the Text-to-Speech API), then make ingest-corpus - or drop a recording in under that name. The cell then runs.")


## Cell 8: Retry what is transient


In [ ]:
import random
from google.genai import errors

# RETRY WHAT IS TRANSIENT, RAISE WHAT IS NOT. 429 and 503 are the platform's, and waiting works; a 400
# is yours, and waiting turns one wrong request into four. genai raises errors.APIError with .code - the
# unified SDK's exception, not api_core's, which is the google-cloud clients' (9.3).
def analyze_with_retry(contents, max_retries: int = 3):
    for attempt in range(max_retries):
        try:
            return gen.models.generate_content(model="gemini-3.6-flash", contents=contents)
        except errors.APIError as e:
            if e.code == 429:                                   # rate limited: back off with jitter
                wait = (2 ** attempt) + random.uniform(0, 1)
                print(f"rate limited - waiting {wait:.1f}s"); time.sleep(wait)
            elif e.code == 503:
                print("service unavailable - waiting 10s"); time.sleep(10)
            else:                                               # 400, 403, 404: not a retry
                raise
    raise RuntimeError("max retries exceeded")

out = analyze_with_retry([types.Part.from_uri(file_uri=INV, mime_type="image/png"), "Who is the invoice billed to? One line."])
print(out.text.strip()[:160])
assert "ACME" in out.text


## Cell 9: Cost, from measured tokens


In [ ]:
# THE MONTHLY ESTIMATE, FROM MEASURED NUMBERS. img_tokens and pdf_tokens came from count_tokens on
# real assets a few cells ago; the per-minute video figure is the old table's unless this run measured
# one. Every scenario is a DocuMind workload - invoices, contracts, town halls, triage.
per_page = pdf_tokens / POSH_PAGES
video_per_min = (vt / 3) if HAS_VIDEO else 18_000        # the synthetic town hall is about three minutes
PRICES = {"flash": 1.50, "flash-lite": 0.25}                # USD per 1M input tokens, standard rates
USD_INR = 85
scenarios = [
    ("500 invoices as page images",        500 * img_tokens,            "flash"),
    ("50 contracts, 30 scanned pages each", 50 * 30 * per_page,         "flash"),
    ("20 town halls, 45 minutes each",     20 * 45 * video_per_min,     "flash"),
    ("100 triage classifications",         100 * 258,                    "flash-lite"),
]
print(f"{'workload':40} {'tokens':>12} {'model':<10} {'USD':>9} {'INR':>9}")
print("-" * 84)
total = 0.0
for name, tokens, model in scenarios:
    usd = tokens * PRICES[model] / 1_000_000
    total += usd
    print(f"{name:40} {int(tokens):>12,} {model:<10} {usd:>9.3f} {usd * USD_INR:>9.0f}")
print("-" * 84)
print(f"{'TOTAL / month':40} {'':>12} {'':<10} {total:>9.2f} {total * USD_INR:>9.0f}")
print("\nand the lane's own number: a question about the scan costs one CHUNK, because the 13 pages were read once at ingest.")


## Where this goes
- **9.2** generates media through the Studio's route and reads answers aloud; **9.3** puts Vision's OCR of a real page beside Document AI's.
- **9.4** is the Studio end to end, including the signed upload that becomes an ingest; **9.6** is the citation contract these figures come back through.

## ✅ Lesson 9.1 complete
- ✅ An image read by reference and checked against the corpus's text
- ✅ Tokens measured on a real page and a real scan; cost from the measurement
- ✅ The scan read directly and through the lane: 13 pages a question, or one chunk
- ✅ Two buckets, one rule; nothing created from a notebook
- ✅ Two captions for one figure, and the retriever's is the one that counts
- ✅ Video by URI at LOW resolution, when the corpus holds one
